## Tutorial for Applying SoccerCPD to kloppy Data

In [1]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd
import rpy2.robjects as robjects
import rpy2.robjects.packages as rpackages

from src.match import Match
from src.kloppy import Kloppy
from src.soccercpd import SoccerCPD

In [ ]:
robjects.r(".libPaths('/usr/lib/R/site-library')")
utils = rpackages.importr('utils')
utils.chooseCRANmirror(ind=1)
if not rpackages.isinstalled('gSeg'):
    utils.install_packages('gSeg')
rpackages.importr('gSeg')

### Converting preprocessed RGP data into SoccerCPD format

In [ ]:
home_id = 82577
away_id = 82579
data = pd.read_csv(f"data/kloppy/{home_id}-{away_id}.csv", header=0, parse_dates=["datetime"])
col_dict = {"index": "frame_id", "session": "period_id", "time": "timestamp"}
data = data.reset_index().rename(columns=col_dict)
data

In [ ]:
kloppy = Kloppy(data)
kloppy.rotate_pitch()
input_data = kloppy.convert_to_soccercpd_input(exclude_gks=False)
input_data

### Converting kloppy data into SoccerCPD format

In [ ]:
data = pd.read_pickle("data/kloppy/sample_kloppy_file.pkl")
data

In [ ]:
kloppy = Kloppy(data)
kloppy.preprocess_times()
kloppy.round_change_times()
kloppy.label_player_periods()
input_data = kloppy.convert_to_soccercpd_input()
input_data

### Formation detection per team

In [ ]:
home_away = "home"  # Choose either home or away
team_data = input_data[input_data["team"] == home_away]
cpd = SoccerCPD(team_data)
cpd.run()

### Label assignment using precomputed formation clusters

In [ ]:
benchmark_forms = pd.read_pickle("data/benchmark_forms.pkl")
benchmark_roles = pd.read_csv("data/benchmark_roles.csv", header=0)
# cpd.label_roles(role_benchmarks, form_benchmarks)

form_labels = {1: "4231", 2: "442"}
cpd.label_roles(benchmark_roles, form_labels=form_labels)
cpd.role_labels

### Saving results and visualization

In [7]:
activity_id = home_id if home_away == "home" else away_id

In [ ]:
cpd.save_stats(match_id=activity_id)

In [ ]:
roster = pd.read_csv(f"data/roster/{activity_id}.csv", index_col=0, header=0)
cpd.visualize(activity_id, roster=roster, role_labels=cpd.role_labels, save=True)

### Calculating and visualizing stats per player-role pair

In [12]:
match = Match(cpd.data, cpd.role_summary)
match.compute_stats()

In [14]:
import math
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
match.plot_by_player("distance")

In [ ]:
match.plot_by_player("hsr_dist")

In [ ]:
match.compute_role_stats()
match.plot_by_role("distance_90min")